# Review Manhwa

Cắt panel, đọc chữ, viết lời review tiếng Việt, đọc bằng **VieNeu-TTS v3 Turbo**, rồi ghép video. Giao diện web hiện ngay trong notebook.

Trước khi chạy: **Runtime → Change runtime type → T4 GPU**.

Repo private thì vào Colab → **Secrets**, thêm secret tên `GITHUB_TOKEN` (quyền đọc nội dung repo) và bật **Notebook access**.

Chỉ tải lên truyện bạn có quyền sử dụng. Tool không lấy chương từ website.

In [ ]:
import os
import shutil
import subprocess
import sys

ROOT = "/content/review-manhwa"
REPO = "https://github.com/hiepit127/review-manhwa.git"

def _token():
    try:
        from google.colab import userdata
        return (userdata.get("GITHUB_TOKEN") or "").strip()
    except Exception:
        return os.environ.get("GITHUB_TOKEN", "").strip()

if not os.path.exists("app.py"):
    if not os.path.exists(os.path.join(ROOT, "app.py")):
        if os.path.isdir(ROOT):
            shutil.rmtree(ROOT)
        env = os.environ.copy()
        env["GIT_TERMINAL_PROMPT"] = "0"
        cmd = ["git", "clone", "--depth", "1"]
        token = _token()
        if token:
            cmd += ["-c", f"http.extraHeader=AUTHORIZATION: bearer {token}"]
        cmd += [REPO, ROOT]
        cloned = subprocess.run(cmd, env=env).returncode == 0
        if not cloned or not os.path.exists(os.path.join(ROOT, "app.py")):
            raise SystemExit(
                "Không clone được repo. Nếu repo private, vào Colab → Secrets, "
                "thêm GITHUB_TOKEN (quyền đọc repo), bật notebook access, rồi chạy lại ô này."
            )
    os.chdir(ROOT)

if not os.path.exists("requirements-colab.txt"):
    raise SystemExit(f"Không thấy requirements-colab.txt trong {os.getcwd()}")

if os.path.exists("/content"):
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg", "fonts-dejavu-core"], check=True)

keep = []
try:
    import importlib.metadata as metadata
    for name in ("torch", "torchvision", "numpy"):
        try:
            keep.append(f"{name}=={metadata.version(name)}")
        except metadata.PackageNotFoundError:
            pass
except Exception:
    pass
constraint = "/tmp/review-manhwa-keep.txt"
with open(constraint, "w", encoding="utf-8") as handle:
    handle.write("\n".join(keep) + "\n")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt", "-c", constraint],
    check=True,
)
subprocess.run([sys.executable, "app.py", "--doctor"], check=True)

try:
    import torch
    print("CUDA", torch.cuda.is_available(), torch.__version__)
    if not torch.cuda.is_available():
        print("Chưa thấy GPU. Chọn Runtime → Change runtime type → T4, rồi chạy lại ô này.")
except Exception as error:
    print(error)


In [ ]:
import os
import sys

sys.path.insert(0, os.getcwd())
from app import launch

launch()
